In [0]:
import io, zipfile, requests
from datetime import date

spark.sql("CREATE VOLUME IF NOT EXISTS gwinnett.bronze.raw_files")

URL = "https://www.gwinnettcounty.com/content/departments/its/gis-maps/tax_assessor_property_ownership_data.zip"
q = f"{date.today().year}Q{(date.today().month - 1) // 3 + 1}"
dest = f"/Volumes/gwinnett/bronze/raw_files/{q}"

resp = requests.get(URL, timeout=300)
resp.raise_for_status()
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    name = next(n for n in zf.namelist() if n.endswith("Property Ownership.xlsx"))  # the year in the filename changes
    zf.extract(name, dest)

In [0]:
import glob
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
!pip install openpyxl

path = glob.glob(f"{dest}/**/*Property Ownership.xlsx", recursive=True)[0]
pdf = pd.read_excel(path, dtype=str, engine='openpyxl')  # all text; Silver does the casting
pdf = pdf.where(pdf.notna(), None)                         # blank cells become real nulls

schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
sdf = (spark.createDataFrame(pdf, schema=schema)
       .withColumn("source_quarter", F.lit(q))
       .withColumn("ingested_at", F.current_timestamp()))

In [0]:
(sdf.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"source_quarter = '{q}'")   # re-running replaces this quarter only
    .saveAsTable("gwinnett.bronze.property_ownership_raw"))

print(spark.table("gwinnett.bronze.property_ownership_raw").count())

In [0]:
print(spark.table("gwinnett.bronze.property_ownership_raw").count())